In [35]:
import json

In [44]:
import re
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

@dataclass
class Span:
    type: str
    value: str
    start: int
    end: int

# ----------------------------
# Regex patterns (tunable)
# ----------------------------
RE_URL = re.compile(
    r'\b(?:https?://|ftp://|www\.)[^\s<>"\)\]]+',
    re.IGNORECASE
)

RE_EMAIL = re.compile(
    r'\b[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}\b',
    re.IGNORECASE
)

# Strict-ish IPv4 (0-255 each octet)
RE_IPV4 = re.compile(
    r'\b(?:(?:25[0-5]|2[0-4]\d|1?\d?\d)\.){3}'
    r'(?:25[0-5]|2[0-4]\d|1?\d?\d)\b'
)
RE_IPV6 = re.compile(
    r'\b(?:'
    r'(?:[0-9a-f]{1,4}:){7}[0-9a-f]{1,4}'
    r'|(?:[0-9a-f]{1,4}:){1,7}:'
    r'|:(?::[0-9a-f]{1,4}){1,7}'
    r'|(?:[0-9a-f]{1,4}:){1,6}:[0-9a-f]{1,4}'
    r')\b',
    re.IGNORECASE
)

RE_NAMED_PIPE = re.compile(r'\\\\\.\\pipe\\[^\s,;:"\'\)\]\}<>]+', re.IGNORECASE)
RE_ADMIN_SHARE = re.compile(r'\\\\[^\s\\]+\\(?:ADMIN\$|C\$|IPC\$)(?:\\[^\s,;:"\'\)\]\}<>]+)?', re.IGNORECASE)

# CVE IDs like CVE-2017-11882
RE_CVE = re.compile(
    r'\bCVE-\d{4}-\d{4,7}\b',
    re.IGNORECASE
)

# Common hashes: MD5(32), SHA1(40), SHA256(64), SHA512(128)
RE_FILE_HASH = re.compile(
    r'\b(?:[a-f0-9]{32}|[a-f0-9]{40}|[a-f0-9]{64}|[a-f0-9]{128})\b',
    re.IGNORECASE
)

# File path: Windows or Unix, bounded (won't swallow the rest of the sentence)
RE_FILE_PATH = re.compile(
    r'('
    # Windows path like C:\Users\bob\AppData\Local\Temp\evil.exe
    r'\b[a-zA-Z]:\\[^\s,;:"\'\)\]\}<>]+'
    r'|'
    # UNC path like \\server\share\file.exe
    r'\\\\[^\s,;:"\'\)\]\}<>]+'
    r'|'
    # Unix path like /home/user/file.o
    r'/(?:[^\s,;:"\'\)\]\}<>]+)'
    r')'
)

RE_MAC = re.compile(r'\b(?:[0-9a-f]{2}[:-]){5}[0-9a-f]{2}\b', re.IGNORECASE)

# Registry keys (very common prefixes)
RE_REGKEY = re.compile(
    r'\b(?:HKLM|HKCU|HKCR|HKU|HKCC)\\[^\s,;"]+',
    re.IGNORECASE
)

# Domains (keep conservative; avoid catching file extensions too often)
# Note: This can still false-positive on "something.exe" if you aren't careful.
RE_DOMAIN = re.compile(
    r'\b(?:[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?\.)+'
    r'(?:[a-z]{2,63})\b',
    re.IGNORECASE
)

# Filenames (conservative extensions list; add more as needed)
RE_FILENAME = re.compile(
    r'\b[\w\-]+\.(?:vba|ps1|bat|cmd|exe|dll|sys|js|vbs|docm|xlsm|pptm|pdf|zip|rar|7z|tar|gz|py|sh|o|obj)\b',
    re.IGNORECASE
)

# ----------------------------
# Gazetteers (example lists)
# Expand with your paper lists
# ----------------------------
DEFAULT_GAZETTEERS: Dict[str, List[str]] = {
    "ENC_ALGO": [
        "base64", "xor", "aes", "rsa", "rc4", "des", "3des", "chacha20",
        "blowfish", "twofish", "md5", "sha1", "sha256"  # include if you want algo mentions too
    ],
    "PROTOCOL": [
        "http", "https", "smtp", "imap", "pop3", "dns", "dhcp", "ftp", "sftp",
        "ssh", "rdp", "smb", "ldap", "kerberos", "ntp", "snmp", "tls", "tcp", "udp"
    ],
    "DATA_OBJECT": [
        "clipboard", "screen", "password", "credentials", "keystroke", "keystrokes",
        "cookie", "cookies", "token", "tokens", "registry", "file", "files"
    ],
}
def normalize_defanging(text: str) -> str:
    text = re.sub(r'\bhxxps?\b', lambda m: 'https' if m.group(0).lower() == 'hxxps' else 'http', text, flags=re.IGNORECASE)
    text = text.replace('[.]', '.').replace('(.)', '.').replace('{.}', '.')
    text = text.replace('[:]', ':').replace('[://]', '://')
    return text

def _compile_gazetteer_terms(terms: List[str]) -> re.Pattern:
    # sort longest-first to prefer multiword/longer matches
    escaped = sorted((re.escape(t) for t in terms), key=len, reverse=True)
    # \b boundaries help but multiword tokens still work; adjust if you want partial matches
    return re.compile(r'\b(?:' + '|'.join(escaped) + r')\b', re.IGNORECASE)

class TTPPreprocessor:
    def __init__(
        self,
        gazetteers: Optional[Dict[str, List[str]]] = None,
        replace_with_placeholders: bool = True
    ):
        self.replace_with_placeholders = replace_with_placeholders
        self.gazetteers = gazetteers or DEFAULT_GAZETTEERS

        self.regex_detectors: List[Tuple[str, re.Pattern]] = [
            ("URL", RE_URL),
            ("EMAIL", RE_EMAIL),
            ("IPv4", RE_IPV4),
            ("IPv6", RE_IPV6),
            ("CVE", RE_CVE),
            ("FILE_HASH", RE_FILE_HASH),
            ("FILE_PATH", RE_FILE_PATH),
            ("REGKEY", RE_REGKEY),
            ("FILENAME", RE_FILENAME),
            ("DOMAIN", RE_DOMAIN),
            ("MAC", RE_MAC),
            ("NAMED_PIPE", RE_NAMED_PIPE),
            ("ADMIN_SHARE", RE_ADMIN_SHARE),
        ]

        self.gazetteer_detectors: List[Tuple[str, re.Pattern]] = [
            ("ENC_ALGO", _compile_gazetteer_terms(self.gazetteers["ENC_ALGO"])),
            ("PROTOCOL", _compile_gazetteer_terms(self.gazetteers["PROTOCOL"])),
            ("DATA_OBJECT", _compile_gazetteer_terms(self.gazetteers["DATA_OBJECT"])),
        ]

    def extract_spans(self, text: str) -> List[Span]:
        spans: List[Span] = []

        # 1) regex spans
        for typ, pat in self.regex_detectors:
            for m in pat.finditer(text):
                spans.append(Span(typ, m.group(0), m.start(), m.end()))

        # 2) gazetteer spans
        for typ, pat in self.gazetteer_detectors:
            for m in pat.finditer(text):
                spans.append(Span(typ, m.group(0), m.start(), m.end()))

        # Resolve overlaps: keep longest span; if tie, keep earlier
        spans.sort(key=lambda s: (s.start, -(s.end - s.start)))
        merged: List[Span] = []
        for s in spans:
            if not merged:
                merged.append(s)
                continue
            prev = merged[-1]
            if s.start < prev.end:
                # overlap: keep the longer one
                prev_len = prev.end - prev.start
                s_len = s.end - s.start
                if s_len > prev_len:
                    merged[-1] = s
                # else discard s
            else:
                merged.append(s)

        return merged

    def transform(self, text: str) -> Tuple[str, List[Span]]:
        text = normalize_defanging(text)
        spans = self.extract_spans(text)

        if not spans:
            return text, []

        # rebuild string from left to right
        out = []
        last = 0
        for s in spans:
            out.append(text[last:s.start])
            if self.replace_with_placeholders:
                out.append(f"<{s.type}>")
            # else: drop the matched value entirely
            last = s.end
        out.append(text[last:])

        # cleanup: normalize whitespace
        clean = re.sub(r'\s+', ' ', ''.join(out)).strip()
        return clean, spans

# ----------------------------
# Example
# ----------------------------
if __name__ == "__main__":
    p = TTPPreprocessor(replace_with_placeholders=True)

    sent = (
        "Download http://example.com/project/example.php and email mail@example.com. "
        "Dropper writes to C:\\Users\\bob\\AppData\\Local\\Temp\\evil.exe, "
        "sets HKCU\\Software\\Microsoft\\Windows\\CurrentVersion\\Run, "
        "connects over SMTP to 192.168.1.1, uses Base64 and XOR, references CVE-2017-11882, "
        "hash 66efff4c945d3c3b87fc271b47d456db."
    )

    clean, spans = p.transform(sent)
    print(clean)


Download <URL> and email <EMAIL>. Dropper writes to <FILE_PATH>, sets <REGKEY>, connects over <PROTOCOL> to <IPv4>, uses <ENC_ALGO> and <ENC_ALGO>, references <CVE>, hash <FILE_HASH>.


In [45]:
import json
from copy import deepcopy

# --- assume you already have TTPPreprocessor defined (from earlier) ---
# from your_module import TTPPreprocessor

def should_preprocess(labels_for_sentence, prefix="T1") -> bool:
    """
    labels_for_sentence: list like ["T1210", "G0007"] or []
    """
    if not labels_for_sentence:
        return False
    return any(isinstance(l, str) and l.startswith(prefix) for l in labels_for_sentence)

def preprocess_json(
    input_path: str,
    output_path: str,
    label_prefix: str = "T1",
    replace_with_placeholders: bool = True,
    store_spans: bool = True
):
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    pre = TTPPreprocessor(replace_with_placeholders=replace_with_placeholders)

    out = deepcopy(data)

    # Your structure is dict-of-dicts keyed by string IDs ("0","1","2"...)
    sent_dict = out.get("sentence", {})
    labels_dict = out.get("labels", {})

    # Where to store preprocessing metadata (optional)
    if store_spans and "preprocess_spans" not in out:
        out["preprocess_spans"] = {}

    changed = 0
    for sid, sent in sent_dict.items():
        labs = labels_dict.get(sid, [])
        clean, spans = pre.transform(sent)
        sent_dict[sid] = clean
        changed += 1

        if store_spans:
            out["preprocess_spans"][sid] = [
                {"type": s.type, "value": s.value, "start": s.start, "end": s.end}
                for s in spans
            ]

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)

    print(f"Done. Preprocessed {changed} sentences (label prefix='{label_prefix}').")


In [49]:
preprocess_json("/home/simonettos/thijs/classification/classification/datasets/tram_train.json", "/home/simonettos/thijs/classification/classification/datasets/tram_train_preprocessed.json", label_prefix="T1")

Done. Preprocessed 15358 sentences (label prefix='T1').
